# Intervention Mix Effectiveness

## 1) Problem framing
**Business question:** Which intervention combinations are associated with healthier emotional session outcomes and lower escalation risk?

- Explanatory goal: estimate intervention effects on emotional delta with **resident context** (case mix, referral, risk) and **prior-session history** before each session date.
- Predictive goal: predict `concerns_flagged` from **pre-session** features only (history strictly before `session_date`).
- Success metrics: explanatory R2/MAE; predictive ROC-AUC / F1 under resident-group holdout.

**History policy:** Prior session counts, mean concerns, mean duration, and lagged emotional bucket use only recordings **before** the current row’s `session_date` for that `resident_id`.

Deployment candidate: `/api/reports/trends/intervention-mix` and counselor supervisor triage panel.


In [ ]:

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LinearRegression
from sklearn.metrics import f1_score, mean_absolute_error, r2_score, roc_auc_score
from sklearn.model_selection import GroupKFold, GroupShuffleSplit, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

import sys
from pathlib import Path
for candidate in [Path.cwd(), Path.cwd() / 'ml-pipelines', Path.cwd().parent]:
    if (candidate / 'data_loader.py').exists():
        sys.path.insert(0, str(candidate))
        break
from data_loader import load_table

p = load_table('process_recordings').sort_values('session_date').copy()
res = load_table('residents')[
    ['resident_id', 'safehouse_id', 'case_category', 'referral_source', 'current_risk_level']
]
p = p.merge(res, on='resident_id', how='left')

emotion_map = {'Distressed': 1, 'Withdrawn': 2, 'Sad': 2, 'Anxious': 2, 'Angry': 2, 'Calm': 4, 'Hopeful': 5, 'Happy': 5}
p.loc[:, 'emo_start'] = p['emotional_state_observed'].map(emotion_map).fillna(3)
p.loc[:, 'emo_end'] = p['emotional_state_end'].map(emotion_map).fillna(3)
p.loc[:, 'emo_delta'] = (p['emo_end'] - p['emo_start']).clip(-3, 3)
p.loc[:, 'intervention_count'] = p['interventions_applied'].fillna('').str.count(',') + 1
p.loc[p['interventions_applied'].fillna('') == '', 'intervention_count'] = 0
p.loc[:, 'high_concern'] = p['concerns_flagged'].astype(int)
p.loc[:, 'is_individual'] = (p['session_type'] == 'Individual').astype(int)
p.loc[:, 'is_group'] = (p['session_type'] == 'Group').astype(int)
p.loc[:, 'group_x_intervention_ct'] = p['is_group'] * p['intervention_count']

# Top intervention tokens (comma-separated labels)
tok_series = p['interventions_applied'].fillna('').str.split(',').apply(
    lambda xs: [t.strip() for t in xs if t.strip()]
)
all_toks = [t for row in tok_series for t in row]
top_toks = list(pd.Series(all_toks).value_counts().head(6).index)
for i, t in enumerate(top_toks):
    p.loc[:, f'tok_w{i}'] = p['interventions_applied'].fillna('').str.contains(t, regex=False).astype(int)


def enrich_history(sub: pd.DataFrame) -> pd.DataFrame:
    sub = sub.sort_values('session_date').copy()
    prior_n = []
    prior_concern = []
    prior_dur = []
    lag1_emo = []
    for _, row in sub.iterrows():
        t = row['session_date']
        past = sub[sub['session_date'] < t]
        prior_n.append(len(past))
        prior_concern.append(float(past['concerns_flagged'].astype(int).mean()) if len(past) else 0.0)
        prior_dur.append(float(past['session_duration_minutes'].mean()) if len(past) else float(row['session_duration_minutes']))
        lag1_emo.append(float(past['emo_start'].iloc[-1]) if len(past) else 3.0)
    sub.loc[:, 'prior_session_count'] = prior_n
    sub.loc[:, 'prior_mean_concern'] = prior_concern
    sub.loc[:, 'prior_mean_duration'] = prior_dur
    sub.loc[:, 'lag1_emo_start'] = lag1_emo
    return sub

parts = [enrich_history(g) for _, g in p.groupby('resident_id', sort=False)]
p = pd.concat(parts, ignore_index=True)

base_feats = [
    'session_duration_minutes', 'is_individual', 'is_group', 'emo_start', 'intervention_count',
    'group_x_intervention_ct', 'social_worker', 'safehouse_id', 'case_category', 'referral_source',
    'current_risk_level', 'prior_session_count', 'prior_mean_concern', 'prior_mean_duration', 'lag1_emo_start',
]
tok_cols = [c for c in p.columns if c.startswith('tok_w')]
features = base_feats + tok_cols
X = p[features].copy()
y_reg = p['emo_delta']
y_clf = p['high_concern']

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
idx_tr, idx_te = next(gss.split(X, y_clf, groups=p['resident_id']))
Xtr, Xte = X.iloc[idx_tr], X.iloc[idx_te]
ytr_reg, yte_reg = y_reg.iloc[idx_tr], y_reg.iloc[idx_te]
ytr_clf, yte_clf = y_clf.iloc[idx_tr], y_clf.iloc[idx_te]

cat_cols = [c for c in X.columns if X[c].dtype == 'object' or str(X[c].dtype) == 'bool']
num_cols = [c for c in X.columns if c not in cat_cols]
prep = ColumnTransformer([
    ('num', Pipeline([('impute', SimpleImputer(strategy='median')), ('scale', StandardScaler())]), num_cols),
    ('cat', Pipeline([('impute', SimpleImputer(strategy='most_frequent')), ('oh', OneHotEncoder(handle_unknown='ignore'))]), cat_cols)
])

lin = Pipeline([('prep', prep), ('model', LinearRegression())])
lin.fit(Xtr, ytr_reg)
pred_reg = lin.predict(Xte)
print('Explanatory R2:', round(r2_score(yte_reg, pred_reg), 3))
print('Explanatory MAE:', round(mean_absolute_error(yte_reg, pred_reg), 3))

baseline = Pipeline([('prep', prep), ('model', DummyClassifier(strategy='prior'))])
rf = Pipeline([('prep', prep), ('model', RandomForestClassifier(
    n_estimators=280, random_state=42, min_samples_leaf=4, class_weight='balanced_subsample'))])
baseline.fit(Xtr, ytr_clf)
rf.fit(Xtr, ytr_clf)
base_proba = baseline.predict_proba(Xte)[:, 1]
rf_proba = rf.predict_proba(Xte)[:, 1]
rf_pred = (rf_proba >= 0.5).astype(int)
print('Baseline AUC:', round(roc_auc_score(yte_clf, base_proba), 3))
print('RF AUC:', round(roc_auc_score(yte_clf, rf_proba), 3))
print('RF F1:', round(f1_score(yte_clf, rf_pred, zero_division=0), 3))

gkf = GroupKFold(n_splits=min(5, p['resident_id'].nunique()))
cv = cross_validate(
    rf, X, y_clf, cv=gkf, scoring=['roc_auc', 'f1'], groups=p['resident_id'].values, n_jobs=-1,
)
print('CV AUC mean/std:', round(float(cv['test_roc_auc'].mean()), 3), round(float(cv['test_roc_auc'].std()), 3))

perm = permutation_importance(rf, Xte, yte_clf, n_repeats=8, random_state=42, scoring='roc_auc')
imp = pd.Series(perm.importances_mean, index=X.columns).sort_values(ascending=False).head(12)
print('\nTop predictive drivers:')
print(imp.round(4).to_string())

coef_values = np.ravel(lin.named_steps['model'].coef_)
coef_names = lin.named_steps['prep'].get_feature_names_out()
usable = min(len(coef_values), len(coef_names))
coef = pd.Series(coef_values[:usable], index=coef_names[:usable]).sort_values(key=lambda s: s.abs(), ascending=False).head(12)
print('\nTop explanatory effects:')
print(coef.round(4).to_string())

print('\nDecision recommendations: route high predicted concern sessions to supervisors; prioritize bundles where history + session context show stable gains.')


In [ ]:
import sys
from pathlib import Path
for candidate in [Path.cwd(), Path.cwd() / 'ml-pipelines', Path.cwd().parent]:
    if (candidate / 'trend_eval_helpers.py').exists():
        sys.path.insert(0, str(candidate))
        break
from trend_eval_helpers import print_calibration_bins, print_threshold_scan, bootstrap_linear_coefs, fairness_binary, fairness_regression_mae

print("\n=== Evaluation artifacts ===")
if yte_clf.nunique() >= 2:
    proba = rf.predict_proba(Xte)[:, 1]
    print_calibration_bins(yte_clf.values, proba)
    print_threshold_scan(yte_clf.values, proba)
    fairness_binary(rf, Xte, yte_clf, p.loc[Xte.index], ['safehouse_id', 'case_category'], min_n=20)
bootstrap_linear_coefs(lin, Xtr, ytr_reg, n_boot=150, top_k=8)
